In [1]:
from google.colab import drive
drive.flush_and_unmount()
#in case of breakage, unmounts drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
#mounts google drive

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/CITY_FMP
#enters current working directory of project

/content/drive/MyDrive/CITY_FMP


In [4]:
!ls

 config.py	   'disease data'     'Tomato Training Output'	 yolo11n.pt
 config.yaml	    logger.py	       trainDetector.py		 yolo12n.pt
 controller.ipynb   __pycache__        train.py			 yolov12
 dataset	    ReplacementFiles   utilities.py		 yolov8n.pt
 dataset.py	    requirements.txt   wandb


In [5]:
!python --version

Python 3.12.11


In [6]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Mon Sep  8 10:28:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8       

In [7]:
!pip install -r requirements.txt

In [1]:
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/12/yolo12.yaml
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/cfg/default.yaml
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/head.py
!ls /usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py

#confirms all required files are present in the ultralytics package. Note: ultralytics 8.3.161 must be used

'ls' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.
'ls' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
#replaces necessary file components automatically
# file_path_trainer = os.path.join(os.getcwd(), 'yolov12', 'ultralytics', 'engine', 'trainer.py' )
# file_path_default = os.path.join(os.getcwd(), 'yolov12', 'ultralytics', 'cfg', 'defaut' )

#sets files paths to the specified paths in the ultralytics downloaded package
file_path_trainer = '/usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py'
file_path_default = '/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/default.yaml'
file_path_conv = '/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py'
file_path_v12 = '/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/models/12/yolo12.yaml'
file_path_head = '/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/head.py'
file_path_loss = '/usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py'

#opens the new files and saves their contents
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/trainer.py', 'r') as trainer:
    replace_trainer = str(trainer.read())
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/default.yaml', 'r') as default:
    replace_default = str(default.read())
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/conv.py', 'r') as conv:
    replace_conv = str(conv.read())
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/yolov12.yaml', 'r') as v12:
    replace_v12 = str(v12.read())
with open('/content/drive/MyDrive/CITY_FMP/ReplacementFiles/loss.py', 'r') as loss:
    replace_loss = str(loss.read())

#head file needed to be treated slightly differently due to file incompatabilities, the class is added onto the file at the end  
new_head = """
class ReDesignedDetectionHead(nn.Module):
    stride = None
    dynamic = False
    export = False

    def __init__(self, nc=80, ch=(), groups=4):
        super().__init__()
        self.nc = nc
        self.no = nc + 5
        self.nl = len(ch)

        self.shared1 = nn.Conv2d(ch[0], ch[0], 3, 1, 1, groups=groups, bias=False)
        self.shared2 = nn.Conv2d(ch[0], ch[0], 3, 1, 1, groups=groups, bias=False)

        self.proj = nn.ModuleList([nn.Conv2d(c, ch[0], 1, 1, 0) for c in ch])
        self.m = nn.ModuleList([nn.Conv2d(ch[0], self.no, 1, 1, 0) for _ in ch])

    def forward(self, x):
        z = []
        for i, f in enumerate(x):
            f = self.proj[i](f)
            f = self.shared1(f)
            f = self.shared2(f)
            z.append(self.m[i](f))
        return z
"""

#replaces the ultralytics files with improved custom files
#uncomment commented code for AHG-YOLO file replacement
with open(file_path_trainer, 'w') as fT:
    fT.write(replace_trainer)
    print("Trainer successfully replaced")
with open(file_path_default, 'w') as fD:
    fD.write("---\n" + replace_default)
    print("Default successfully replaced")
# with open(file_path_conv, 'w') as fC:
#     fC.write(replace_conv)
#     print("Conv successfully replaced")
# with open(file_path_v12, 'w') as fV:
#     fV.write(replace_v12)
#     print("YOLOv12 successfully replaced")
# with open(file_path_head, 'a') as fH:
#     fH.write("\n" + new_head)
#     print("Head successfully replaced")
# with open(file_path_loss, 'w') as fL:
#     fL.write(replace_loss)
#     print("Loss successfully replaced")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CITY_FMP/ReplacementFiles/trainer.py'

In [14]:
!yolo settings wandb=True

✅ Updated 'wandb=True'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/content/drive/MyDrive/CITY_FMP/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "569f3ba64b326db489132663f79cd37279811de477381b83ac131e6cdd129cbb",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": true,
  "vscode_msg": true,
  "openvino_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


In [ ]:
!python -u train.py

Selected Device: cuda
Tesla T4
PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA version: 12.6
wandb: Currently logged in as: unliveddisc03 (unliveddisc03-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.21.3
wandb: Run data is saved locally in /content/drive/MyDrive/CITY_FMP/wandb/run-20250908_105905-poe0s1hq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Tomato-YOLOv12-Rebalanced-(median)
wandb: ⭐️ View project at https://wandb.ai/unliveddisc03-city-university-of-london/Final%20Individual%20Project
wandb: 🚀 View run at https://wandb.ai/unliveddisc03-city-university-of-london/Final%20Individual%20Project/runs/poe0s1hq
Dataset File Updated.
Selected weighted dataset
New https://pypi.org/project/ultralytics/8.3.195 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.161 🚀 Python-3.12.1